# Validation on two more same-clothes datasets: iLIDS-VID and PRID-2011

**Purpose:** confirm that our occlusion-aware fingerprints work beyond MEVID. Both datasets are standard **same-clothes video ReID** benchmarks with two cameras each.

| Dataset | People used | Cameras | Notes |
|---|---|---|---|
| iLIDS-VID | all 300 | 2 | Busy airport footage with heavy occlusion |
| PRID-2011 | 200 (those seen by both cameras) | 2 | Outdoor footage, less clutter |

**Exactly the same method and settings as MEVID-SC:**
- No labels and no training. The models are used **zero-shot**: CSCI with its MEVID weights, FastReID and OSNet with their MSMT17 weights.
- Five fingerprints per clip: all frames, **ours (person visible)**, ours simpler (person found), random removal, and feature-distance removal. The two controls remove as many frames as ours.
- Results: accuracy (probe from camera 1, gallery from camera 2) and consistent ID over 50 fair tests.

**Data:** the notebook uses your Drive copies in `Data/iLIDS-VID` and `Data/PRID-2011` if they exist. Otherwise it downloads from the official sources ([iLIDS-VID page](https://xiatian-zhu.github.io/downloads_qmul_iLIDS-VID_ReID_dataset.html), [PRID-2011 page](https://tugraz.at/index.php?id=17766)) and saves a copy. If a download fails, get the file from that page and upload it to the folder named in the error.

**Run:** an A100 or L4; cells 1 to 6, about 1 to 1.5 hours in total. Everything is saved to Drive. **Cell 6 is standalone**, so you can run it any time.

**Cite:** Wang et al., ECCV 2014 (iLIDS-VID) and Hirzer et al., SCIA 2011 (PRID-2011).

In [ ]:
# 1. Setup and data. Uses your Drive copies of iLIDS-VID and PRID-2011
#    (Data/iLIDS-VID, Data/PRID-2011) if present; otherwise downloads
#    them from the official sources and saves a copy to Drive.
#    Both are same-clothes, two-camera video ReID datasets.
import os, glob, json, time, shutil, subprocess, sys, tarfile, zipfile
import warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

SD = '/content/drive/Shareddrives/*'
def find(pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(pats[0])

CSCI_DIR = os.path.dirname(find([f'{SD}/Data/MEVID-results/CSCI/ez_eva02_vid_hybrid_extra_best.pth',
                                 f'{SD}/**/CSCI/ez_eva02_vid_hybrid_extra_best.pth']))
WT = f'{CSCI_DIR}/ez_eva02_vid_hybrid_extra_best.pth'
RES = os.path.dirname(CSCI_DIR)
DATA = os.path.dirname(RES)                     # .../Data
OUT = f'{RES}/Validation-iLIDS-PRID'
os.makedirs(OUT, exist_ok=True)
SETTINGS = {'datasets': {'iLIDS-VID': 'all 300 people, 2 cameras (Wang et al., ECCV 2014)',
                         'PRID-2011': 'the 200 people seen by both cameras, multi_shot '
                                      '(Hirzer et al., SCIA 2011)'},
            'models': 'CSCI (MEVID weights), FastReID SBS R101-IBN (MSMT17), '
                      'OSNet x1_0 (MSMT17); all used zero-shot, unchanged',
            'method': 'identical to MEVID-SC: pose check every 4th frame; keep '
                      'frames with the person found and >= 50% of keypoints '
                      'visible (main) or found (simpler); clips with < 8 good '
                      'frames unchanged; controls remove the same number of frames',
            'check_every': 4, 'pose_model': 'yolo11m-pose.pt', 'imgsz': 320,
            'det_conf': 0.25, 'keypoint_conf': 0.5, 'visible_thr': 0.5,
            'min_kept_frames': 8, 'precision_target': 0.90, 'n_splits': 25,
            'seed': 2027, 'control_seed': 2027,
            'calibrator': 'largest cut with pair precision >= target, '
                          'geomspace(0.001, 1.5, 120)',
            'accuracy': 'probe = camera 1 (cam_a), gallery = camera 2 (cam_b)'}
with open(f'{OUT}/SETTINGS_frozen.json', 'w') as f:
    json.dump(SETTINGS, f, indent=2)
st = SETTINGS
DL = '/content/dl'; RAW = '/content/val_raw'
os.makedirs(DL, exist_ok=True); os.makedirs(RAW, exist_ok=True)


def fetch(urls, out, min_mb=50):
    for u in urls:
        print('Trying', u)
        subprocess.run(['wget', '-q', '-O', out, u])
        if os.path.exists(out) and os.path.getsize(out) > min_mb * 1e6:
            return True
    return False


def valid(f):
    return os.path.exists(f) and (tarfile.is_tarfile(f) or zipfile.is_zipfile(f))


def get_archive(name, exts, urls, mega=None):
    have = [f for e in exts for f in glob.glob(f'{DATA}/{name}/*{e}') if valid(f)]
    if have:
        print(f'{name}: using Drive copy {have[0]}')
        return have[0]
    out = f'{DL}/{name}{exts[0]}'
    ok = (fetch(urls, out) and valid(out)) if urls else False
    if not ok and mega:
        subprocess.run('apt-get -qq install -y megatools > /dev/null', shell=True)
        md = f'{DL}/mega_{name}'
        os.makedirs(md, exist_ok=True)
        subprocess.run(['megadl', '--path', md, mega])
        got = [f for f in glob.glob(f'{md}/*') if valid(f)]
        if got:
            out = f'{DL}/{os.path.basename(got[0])}'
            shutil.move(got[0], out); ok = True
    assert ok, (f'{name} could not be downloaded. Download it from its official '
                f'page and upload the archive to {DATA}/{name}/, then rerun.')
    os.makedirs(f'{DATA}/{name}', exist_ok=True)
    shutil.copy(out, f'{DATA}/{name}/')
    print(f'{name}: downloaded and saved to Drive')
    return out


def extract(arc, dest):
    if os.path.isdir(dest) and os.listdir(dest):
        return
    os.makedirs(dest, exist_ok=True)
    if arc.endswith('.zip'):
        zipfile.ZipFile(arc).extractall(dest)
    else:
        tarfile.open(arc).extractall(dest)


extract(get_archive('PRID-2011', ['.zip'],
                    ['https://cloud.tugraz.at/index.php/s/KtC8rK9io6HxNc2/download']),
        f'{RAW}/PRID-2011')
extract(get_archive('iLIDS-VID', ['.tar', '.tar.gz', '.zip'],
                    ['http://www.eecs.qmul.ac.uk/~xiatian/iLIDS-VID/iLIDS-VID.tar'],
                    mega='https://mega.nz/#!ZzZgwKpQ!nQuTo9h9gRGBQ4l6k1VZDHYXDckfyg0-SKGOhrk5CSk'),
        f'{RAW}/iLIDS-VID')


def clips_ilids():
    out = []
    for cd in sorted(glob.glob(f'{RAW}/iLIDS-VID/**/sequences/cam*', recursive=True)):
        c = int(cd[-1])
        for p in sorted(os.listdir(cd)):
            fs = sorted(glob.glob(f'{cd}/{p}/*.png'))
            if fs:
                out.append((int(p.replace('person', '')), c, fs))
    return out


def clips_prid():
    out = []
    for cd in sorted(glob.glob(f'{RAW}/PRID-2011/**/multi_shot/cam_*', recursive=True)):
        c = 1 if cd.endswith('a') else 2
        for p in sorted(os.listdir(cd)):
            if not p.startswith('person_'):
                continue
            pid_ = int(p.split('_')[1])
            fs = sorted(glob.glob(f'{cd}/{p}/*.png'))
            if pid_ <= 200 and fs:                 # the 200 people in both cameras
                out.append((pid_, c, fs))
    return out


# Repackage each dataset in the MEVID layout CSCI reads (symlinks only):
# ROOT/bbox_test/<pid4>/<name>, test_name.txt, track_test_info.txt,
# query_IDX.txt (probe = camera 1).
DS = {}
for D, fn in [('iLIDS-VID', clips_ilids), ('PRID-2011', clips_prid)]:
    cl = fn()
    if not cl:
        tops = [os.path.relpath(p, RAW) for p in glob.glob(f'{RAW}/{D}/*/*')][:20]
        raise RuntimeError(f'{D}: no clips found; folder layout is {tops}')
    both = set(p for p, c, _ in cl if c == 1) & set(p for p, c, _ in cl if c == 2)
    cl = [x for x in cl if x[0] in both]
    base = f'/content/val/{D}'
    names, info = [], []
    for t, (p, c, fs) in enumerate(cl):
        d = f'{base}/bbox_test/{p:04d}'
        os.makedirs(d, exist_ok=True)
        a = len(names)
        for j, f in enumerate(fs):
            n = f'{p:04d}O001C{c:03d}T{t:04d}F{j:05d}.png'
            if not os.path.lexists(f'{d}/{n}'):
                os.symlink(f, f'{d}/{n}')
            names.append(n)
        info.append([a, len(names) - 1, p, 1, c])
    info = np.array(info, np.int64)
    qidx = np.flatnonzero(info[:, 4] == 1); gidx = np.flatnonzero(info[:, 4] == 2)
    open(f'{base}/test_name.txt', 'w').write('\n'.join(names) + '\n')
    np.savetxt(f'{base}/track_test_info.txt', info, fmt='%d')
    np.savetxt(f'{base}/query_IDX.txt', qidx, fmt='%d')
    np.savez(f'{OUT}/{D}_meta.npz', info=info, qidx=qidx, gidx=gidx,
             names=np.array(names))
    DS[D] = dict(names=names, info=info, qidx=qidx, gidx=gidx, base=base)
    L = info[:, 1] - info[:, 0] + 1
    print(f'{D}: {len(both)} people, {len(info)} clips, {len(names):,} frames, '
          f'clip length {L.min()}-{L.max()} (median {int(np.median(L))})')


In [ ]:
# 2. Pose check on every 4th frame (same model and settings as MEVID-SC).
#    Small datasets: a few minutes. Saved to Drive.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import cv2, torch
from ultralytics import YOLO
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
pose = None
for D, d in DS.items():
    fn = f'{OUT}/{D}_personcheck.npz'
    check = []
    for s, e in d['info'][:, :2]:
        c = list(range(s, e + 1, st['check_every']))
        if c[-1] != e:
            c.append(e)
        check += c
    K = np.array(check)
    if not os.path.exists(fn):
        if pose is None:
            pose = YOLO(st['pose_model'])
        nm, base = d['names'], d['base']

        class Imgs(Dataset):
            def __len__(self): return len(K)
            def __getitem__(self, i):
                n = nm[K[i]]
                return cv2.imread(f'{base}/bbox_test/{n[:4]}/{n}')

        fo = np.zeros(len(K), np.int8); vi = np.zeros(len(K), np.float32); i = 0
        for batch in tqdm(DataLoader(Imgs(), batch_size=128, num_workers=8,
                                     collate_fn=lambda b: b), desc=D):
            for im, r in zip(batch, pose.predict(batch, imgsz=st['imgsz'],
                                                 conf=st['det_conf'], verbose=False)):
                if len(r.boxes):
                    H, W = im.shape[:2]
                    xy = r.boxes.xyxy.cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
                    area = (xy[:, 2] - xy[:, 0]) * (xy[:, 3] - xy[:, 1]) / (H * W)
                    m = int(np.argmax(area * cf))
                    fo[i] = 1
                    vi[i] = (r.keypoints.data.cpu().numpy()[m][:, 2]
                             >= st['keypoint_conf']).mean()
                i += 1
        np.savez(fn, k=K, found=fo, vis=vi)
    z = np.load(fn)
    d.update(K=z['k'], FOUND=z['found'], VIS=z['vis'])
    print(f'{D}: person found in {100 * z["found"].mean():.1f}% of checked frames; '
          f'median visible body share {np.median(z["vis"][z["found"] == 1]):.2f}')


In [ ]:
# 3. Keep-masks: OURS (person visible, the frozen main rule) and the
#    simpler rule (person found). Controls remove as many frames as OURS.
for D, d in DS.items():
    K, FO, VI, info = d['K'], d['FOUND'], d['VIS'], d['info']
    d['RANGES'] = []
    for rule, thr in [('visible', st['visible_thr']), ('found', 0.0)]:
        keep_c = np.ones(len(K), bool); keep_all = []
        for t, (s, e, *_) in enumerate(info):
            lo, hi = np.searchsorted(K, s), np.searchsorted(K, e, 'right')
            if rule == 'visible':
                d['RANGES'].append((lo, hi))
            fr = np.arange(s, e + 1)
            near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
            ok_c = (FO[lo:hi] == 1) & (VI[lo:hi] >= thr) if thr else (FO[lo:hi] == 1)
            keep = ok_c[near - lo]
            if keep.sum() < st['min_kept_frames']:
                keep[:] = True
            else:
                keep_c[lo:hi] = ok_c if ok_c.any() else True
            keep_all.append(keep)
        d[f'KEEP_C_{rule}'], d[f'KEEP_ALL_{rule}'] = keep_c, keep_all
    d['REMOVE_ALL'] = np.array([(~k).sum() for k in d['KEEP_ALL_visible']])
    d['REMOVE_C'] = np.array([(~d['KEEP_C_visible'][lo:hi]).sum()
                              for lo, hi in d['RANGES']])
    tot = len(d['names'])
    for rule in ['visible', 'found']:
        rm = sum((~k).sum() for k in d[f'KEEP_ALL_{rule}'])
        ch = sum((~k).any() for k in d[f'KEEP_ALL_{rule}'])
        print(f'{D}, rule "{rule}": removes {rm:,} of {tot:,} frames '
              f'({100 * rm / tot:.1f}%) in {ch} of {len(info)} clips')


In [ ]:
# 4. Image models (FastReID SBS R101-IBN, OSNet x1_0; MSMT17 weights):
#    frame features on the checked frames, then five fingerprints per clip.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid',
                'gdown', 'yacs', 'termcolor', 'tabulate'], check=True)
import torch.nn.functional as F
import torchvision.transforms as TV
from PIL import Image


def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)


VARIANTS = ['all frames', 'ours (pose: person visible)', 'ours, simpler (person found)',
            'control: random frames', 'control: feature-distance']


def frame_feats(model, tf, dim, d, bs):
    nm, base, K = d['names'], d['base'], d['K']

    class Fr(Dataset):
        def __len__(self): return len(K)
        def __getitem__(self, i):
            n = nm[K[i]]
            return tf(Image.open(f'{base}/bbox_test/{n[:4]}/{n}').convert('RGB'))

    out = np.zeros((len(K), dim), np.float32); i = 0
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for x in DataLoader(Fr(), batch_size=bs, num_workers=8, pin_memory=True):
            f = model(x.cuda(non_blocking=True))
            f = F.normalize((f[0] if isinstance(f, (tuple, list)) else f).float(), dim=1)
            out[i:i + len(f)] = f.cpu().numpy(); i += len(f)
    return out


def fingerprints(Fm, d):
    rng = np.random.default_rng(st['control_seed'])
    out = {v: np.zeros((len(d['info']), Fm.shape[1]), np.float32) for v in VARIANTS}
    for t, (lo, hi) in enumerate(d['RANGES']):
        f = Fm[lo:hi]; k = d['REMOVE_C'][t]
        out[VARIANTS[0]][t] = f.mean(0)
        out[VARIANTS[1]][t] = f[d['KEEP_C_visible'][lo:hi]].mean(0)
        out[VARIANTS[2]][t] = f[d['KEEP_C_found'][lo:hi]].mean(0)
        r = np.ones(hi - lo, bool)
        if k:
            r[rng.choice(hi - lo, k, replace=False)] = False
        out[VARIANTS[3]][t] = f[r].mean(0)
        mu = unit(f.mean(0, keepdims=True))[0]
        q = np.ones(hi - lo, bool)
        if k:
            q[np.argsort(-(1 - f @ mu), kind='stable')[:k]] = False
        out[VARIANTS[4]][t] = f[q].mean(0)
    return out


import torchreid, gdown
m = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000,
                                 loss='softmax', pretrained=True)
ck = os.path.expanduser('~/.cache/torch/checkpoints/osnet_x1_0_msmt17.pth')
os.makedirs(os.path.dirname(ck), exist_ok=True)
for url in ['https://drive.google.com/uc?id=112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M',
            'https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMRmx3MzPwf86x']:
    if not os.path.exists(ck):
        try:
            gdown.download(url, ck, quiet=True, fuzzy=True)
        except Exception as ex:
            print('download failed:', ex)
torchreid.utils.load_pretrained_weights(m, ck)
m = m.cuda().eval()
tf = TV.Compose([TV.Resize((256, 128)), TV.ToTensor(),
                 TV.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
for D, d in DS.items():
    Fm = frame_feats(m, tf, 512, d, 512)
    np.save(f'{OUT}/{D}_osnet_frames.npy', Fm.astype(np.float16))   # for CSCI control
    d['OSN'] = Fm
    np.savez(f'{OUT}/{D}_fingerprints_OSNet.npz', **fingerprints(Fm, d),
             pid=d['info'][:, 2], cam=d['info'][:, 4])
    print(D, 'OSNet done')
del m; torch.cuda.empty_cache()

FRP = '/content/fast-reid'
if not os.path.isdir(FRP):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/JDAI-CV/fast-reid.git', FRP],
                   check=True)
    for f in [f'{FRP}/fastreid/data/build.py', f'{FRP}/fastreid/evaluation/testing.py']:
        s = open(f).read()
        s = s.replace('from collections import Mapping, OrderedDict',
                      'from collections import OrderedDict\nfrom collections.abc import Mapping')
        s = s.replace('from collections import Mapping', 'from collections.abc import Mapping')
        open(f, 'w').write(s)
WFR = '/content/msmt_sbs_R101-ibn.pth'
if not os.path.exists(WFR):
    subprocess.run(['wget', '-q', '-O', WFR, 'https://github.com/JDAI-CV/fast-reid/'
                    'releases/download/v0.1.1/msmt_sbs_R101-ibn.pth'], check=True)
sys.path.insert(0, FRP)
from fastreid.config import get_cfg
from fastreid.modeling.meta_arch import build_model
from fastreid.utils.checkpoint import Checkpointer
cfg = get_cfg(); cfg.merge_from_file(f'{FRP}/configs/MSMT17/sbs_R101-ibn.yml')
cfg.MODEL.BACKBONE.PRETRAIN = False; cfg.freeze()
m = build_model(cfg).cuda().eval(); Checkpointer(m).load(WFR)
tf = TV.Compose([TV.Resize((384, 128)), TV.PILToTensor(), TV.Lambda(lambda t: t.float())])
for D, d in DS.items():
    Fm = frame_feats(m, tf, 2048, d, 256)
    np.savez(f'{OUT}/{D}_fingerprints_FastReID.npz', **fingerprints(Fm, d),
             pid=d['info'][:, 2], cam=d['info'][:, 4])
    print(D, 'FastReID done')
del m; torch.cuda.empty_cache()


In [ ]:
# 5. CSCI (authors' code, MEVID weights, unchanged; same patches as
#    before): five runs per dataset (all frames, ours, simpler, random,
#    feature-distance). Small datasets: a few minutes each. Saved to Drive.
REPO = '/content/ICCV-CSCI-Person-ReID'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ppriyank/'
                    'ICCV-CSCI-Person-ReID.git', REPO], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.9',
                'einops', 'yacs', 'mmengine', 'decord', 'tensorboardX',
                'tensorboard', 'termcolor', 'tabulate', 'h5py', 'scikit-learn',
                'lmdb', 'albumentations', 'torchinfo', 'grad-cam',
                'opencv-python-headless'], check=True)
f = f'{REPO}/processor/eval_mevid.py'; src = open(f).read()
anchor = '    distmat = distmat.cpu()\n'
if 'CSCI_DUMP' not in src:
    assert src.count(anchor) == 1, 'eval_mevid.py changed upstream'
    src = src.replace(anchor, anchor +
        "    if local_rank == 0 and os.environ.get('CSCI_DUMP'):\n"
        "        import numpy as _np\n"
        "        _np.savez(os.environ['CSCI_DUMP'], qf=qf.float().cpu().numpy(), "
        "gf=gf.float().cpu().numpy(), q_pids=q_pids.numpy(), g_pids=g_pids.numpy(), "
        "q_camids=q_camids.numpy(), g_camids=g_camids.numpy())\n")
    if '\nimport os' not in src:
        src = 'import os\n' + src
    open(f, 'w').write(src)
f = f'{REPO}/data/dataset_loader.py'; src = open(f).read()
old = '        self.return_index = return_index\n        self.__getitem__(0)\n'
if old in src:
    open(f, 'w').write(src.replace(old, '        self.return_index = return_index\n'
                                        '        if not train:\n            self.__getitem__(0)\n'))
src = open(f'{REPO}/processor/eval_mevid.py').read()
for loop in ['    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta) in enumerate(dataloader):\n',
             '    for n_iter, (imgs, pids, camids, clothes_id, clothes_ids, meta, index) in enumerate(dataloader):\n']:
    tag = loop + "        if n_iter % 10 == 0 or n_iter == len(dataloader) - 1: print(f'CSCI_PROGRESS {n_iter + 1} {len(dataloader)}', flush=True)\n"
    if loop in src and 'CSCI_PROGRESS' not in src.split(loop)[1][:200]:
        src = src.replace(loop, tag)
open(f'{REPO}/processor/eval_mevid.py', 'w').write(src)
from tqdm.auto import tqdm
TRN = sorted(glob.glob(f'{DATA}/MEVID-SameClothes/annotations/track_train_info.txt'))[0]
TRNN = TRN.replace('track_train_info.txt', 'train_name.txt')
rng = np.random.default_rng(st['control_seed'])
for D, d in DS.items():
    info, names, K = d['info'], d['names'], d['K']
    if 'OSN' not in d:
        d['OSN'] = np.load(f'{OUT}/{D}_osnet_frames.npy').astype(np.float32)
    for v in ['all', 'visible', 'found', 'random', 'feature']:
        OUTF = f'{OUT}/{D}_csci_{v}.npz'
        if os.path.exists(OUTF):
            continue
        root = f'/content/val_root/{D}_{v}'
        os.makedirs(root, exist_ok=True)
        nn_, ni = [], []
        for t, (s, e, p, o, c) in enumerate(info):
            fr = np.arange(s, e + 1); k = d['REMOVE_ALL'][t]
            if v == 'all':
                keep = np.ones(len(fr), bool)
            elif v in ('visible', 'found'):
                keep = d[f'KEEP_ALL_{v}'][t]
            else:
                keep = np.ones(len(fr), bool)
                if k and v == 'random':
                    keep[rng.choice(len(fr), k, replace=False)] = False
                elif k:
                    lo, hi = d['RANGES'][t]
                    near = lo + np.abs(fr[:, None] - K[lo:hi][None, :]).argmin(1)
                    E = d['OSN'][lo:hi]; mu = E.mean(0); mu /= np.linalg.norm(mu)
                    keep[np.argsort(-(1 - d['OSN'][near] @ mu), kind='stable')[:k]] = False
            a = len(nn_)
            nn_ += [names[j] for j in fr[keep]]
            ni.append([a, len(nn_) - 1, p, o, c])
        open(f'{root}/test_name.txt', 'w').write('\n'.join(nn_) + '\n')
        np.savetxt(f'{root}/track_test_info.txt', np.array(ni), fmt='%d')
        np.savetxt(f'{root}/query_IDX.txt', d['qidx'], fmt='%d')
        for n_, s_ in {'train_name.txt': TRNN, 'track_train_info.txt': TRN,
                       'bbox_test': f"{d['base']}/bbox_test"}.items():
            if not os.path.lexists(f'{root}/{n_}'):
                os.symlink(s_, f'{root}/{n_}')
        DUMP = f'/content/csci_dump_{D}_{v}.npz'
        env = dict(os.environ, CSCI_DUMP=DUMP, PYTHONPATH=REPO, PYTHONUNBUFFERED='1')
        cmd = ['torchrun', '--nproc_per_node=1', '--master_port=12381',
               'train_two_step.py', '--eval', '--resume',
               '--config_file', 'configs/mevid_eva02_l_cloth.yml',
               'DATA.ROOT', root + '/', 'TEST.WEIGHT', WT, 'SOLVER.SEED', '12345',
               'TRAIN.TRAIN_VIDEO', 'True', 'DATA.DATASET', 'mevid',
               'MODEL.NAME', 'ez_eva02_vid_hybrid_extra', 'TRAIN.COLOR_PROFILE', '17',
               'DATA.TEST_BATCH', '32']
        t0 = time.time()
        with open(f'{OUT}/{D}_csci_log_{v}.txt', 'w') as log:
            p_ = subprocess.Popen(cmd, cwd=REPO, env=env, text=True,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            bar = None
            for line in p_.stdout:
                log.write(line)
                if line.startswith('CSCI_PROGRESS'):
                    dn, tot = map(int, line.split()[1:3])
                    if bar is None or tot != bar.total or dn < bar.n:
                        if bar is not None:
                            bar.close()
                        bar = tqdm(total=tot, desc=f'{D} {v}')
                    bar.update(dn - bar.n)
                elif any(k_ in line for k_ in ['Error', 'Traceback', 'out of memory']):
                    print(line.rstrip())
            p_.wait()
            if bar is not None:
                bar.close()
        assert os.path.exists(DUMP), f'CSCI stopped before dumping ({D}, {v}); see log'
        z = np.load(DUMP)
        q, g = d['qidx'], d['gidx']
        assert (z['q_pids'] == info[q, 2]).all() and (z['g_pids'] == info[g, 2]).all()
        T = np.zeros((len(info), z['qf'].shape[1]), np.float32)
        T[q], T[g] = z['qf'], z['gf']
        np.savez(OUTF, templates=T, pid=info[:, 2], cam=info[:, 4])
        print(f'{D} {v}: {(time.time() - t0) / 60:.1f} min, saved')


In [ ]:
# 6. STANDALONE results (works after a restart; needs only Drive).
#    Per dataset: accuracy (probe camera 1, gallery camera 2) and
#    consistent ID over 50 fair tests, three models x five variants.
import os, glob, zipfile, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
from tqdm.auto import tqdm
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from scipy.optimize import linear_sum_assignment
from scipy.stats import wilcoxon
drive.mount('/content/drive')
OUT = sorted(glob.glob('/content/drive/Shareddrives/*/Data/MEVID-results/'
                       'Validation-iLIDS-PRID', recursive=True))[0]
st = {'precision_target': 0.90, 'n_splits': 25, 'seed': 2027}
def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-8)
V = ['all frames', 'ours (pose: person visible)', 'ours, simpler (person found)',
     'control: random frames', 'control: feature-distance']
CS = {'all frames': 'all', V[1]: 'visible', V[2]: 'found', V[3]: 'random', V[4]: 'feature'}
MODELS = ['CSCI (MEVID weights)', 'FastReID SBS R101-IBN (MSMT17)', 'OSNet x1_0 (MSMT17)']

def ap_cmc(order, good):
    hits = np.isin(order, good)
    cmc = np.zeros(len(order)); cmc[np.argmax(hits):] = 1.0
    r = np.flatnonzero(hits)
    return np.mean((np.arange(len(r)) + 1) / (r + 1)), cmc

def score(pred, p, c):
    _, pi = np.unique(p, return_inverse=True)
    _, ci = np.unique(pred, return_inverse=True)
    M = np.zeros((pi.max() + 1, ci.max() + 1)); np.add.at(M, (pi, ci), 1)
    r, cc = linear_sum_assignment(-M)
    nc = M.sum(0); tp, same = (M * (M - 1)).sum(), (nc * (nc - 1)).sum()
    _, ki = np.unique(c, return_inverse=True)
    Kc = np.zeros((M.shape[0], ki.max() + 1, M.shape[1])); np.add.at(Kc, (pi, ki, ci), 1)
    nq = M.sum(1)
    x_tot = (nq ** 2 - (Kc.sum(2) ** 2).sum(1)).sum()
    x_same = ((M ** 2).sum(1) - (Kc ** 2).sum((1, 2))).sum()
    return {'IDF1 %': 100 * M[r, cc].sum() / len(p),
            'IDs per person': (M > 0).sum(1).mean(),
            'pair precision %': 100 * tp / same if same else 100.0,
            'cross-camera recall %': 100 * x_same / max(1, x_tot)}

def tree(X, idx):
    D_ = np.clip(1 - X[idx] @ X[idx].T, 0, None); D_ = (D_ + D_.T) / 2
    np.fill_diagonal(D_, 0)
    return linkage(squareform(D_, checks=False), 'average')

CUTS = np.geomspace(0.001, 1.5, 120)
ALL, ALLP = [], []
for D in ['iLIDS-VID', 'PRID-2011']:
    meta = np.load(f'{OUT}/{D}_meta.npz')
    info, qidx, gidx = meta['info'], meta['qidx'], meta['gidx']
    pid, cam = info[:, 2], info[:, 4]

    def calibrate(L, idx):
        best = CUTS[0]
        for t in CUTS:
            if score(fcluster(L, t, 'distance'), pid[idx], cam[idx]
                     )['pair precision %'] >= 100 * st['precision_target']:
                best = t
        return best

    def accuracy(X):
        S = X[qidx] @ X[gidx].T
        aps, r1, r5 = [], [], []
        for i, q in enumerate(qidx):
            good = np.flatnonzero(pid[gidx] == pid[q])
            if good.size:
                a, c = ap_cmc(np.argsort(-S[i]), good)
                aps.append(a); r1.append(c[0]); r5.append(c[min(4, len(c) - 1)])
        return {'mAP %': 100 * np.mean(aps), 'Rank-1 %': 100 * np.mean(r1),
                'Rank-5 %': 100 * np.mean(r5)}

    FEATS = {}
    for v in V:
        fn = f'{OUT}/{D}_csci_{CS[v]}.npz'
        if os.path.exists(fn):
            FEATS[(MODELS[0], v)] = unit(np.load(fn)['templates'])
    for mname, short in zip(MODELS[1:], ['FastReID', 'OSNet']):
        fn = f'{OUT}/{D}_fingerprints_{short}.npz'
        if os.path.exists(fn):
            z = np.load(fn)
            for v in V:
                FEATS[(mname, v)] = unit(z[v])
    people = np.unique(pid)
    rng = np.random.default_rng(st['seed'])
    SPL = []
    for s in range(st['n_splits']):
        perm = rng.permutation(people)
        a = np.isin(pid, perm[:len(people) // 2])
        SPL.append([np.flatnonzero(a), np.flatnonzero(~a)])
    ROWS = []
    for (mname, v), X in tqdm(FEATS.items(), desc=D):
        acc = accuracy(X)
        for s, G in enumerate(SPL):
            Ls = [tree(X, g) for g in G]
            for c, t in [(0, 1), (1, 0)]:
                r = score(fcluster(Ls[t], calibrate(Ls[c], G[c]), 'distance'),
                          pid[G[t]], cam[G[t]])
                r.update(dataset=D, model=mname, variant=v, split=s, direction=c, **acc)
                ROWS.append(r)
    R = pd.DataFrame(ROWS); ALL.append(R)
    for mname in MODELS:
        Rm = R[R.model == mname]
        if (Rm.variant == V[1]).sum() == 0:
            continue
        for ours_v in [V[1], V[2]]:
            o = Rm[Rm.variant == ours_v].set_index(['split', 'direction'])
            refs = [V[0], V[3], V[4]] if ours_v == V[1] else [V[0]]
            for ref in refs:
                b = Rm[Rm.variant == ref].set_index(['split', 'direction'])
                for metric, better in [('IDF1 %', 1), ('IDs per person', -1)]:
                    dd = o[metric] - b[metric]
                    ALLP.append({'dataset': D, 'model': mname.split(' (')[0],
                                 'compare': f'{ours_v} vs {ref}', 'metric': metric,
                                 'mean change': round(dd.mean(), 2),
                                 'better in % of tests': round(100 * (better * dd > 0).mean()),
                                 'p (Wilcoxon)': f'{wilcoxon(dd).pvalue:.1e}'
                                 if (dd != 0).any() else 'n/a'})
REP = pd.concat(ALL)
REP.to_csv(f'{OUT}/val_all_50tests.csv', index=False)
COLS = ['IDF1 %', 'IDs per person', 'cross-camera recall %', 'pair precision %',
        'mAP %', 'Rank-1 %', 'Rank-5 %']
MAIN = REP.groupby(['dataset', 'model', 'variant'], sort=False)[COLS].mean().round(2)
print('Mean over 50 tests:'); display(MAIN)
MAIN.to_csv(f'{OUT}/val_summary.csv')
PAIR = pd.DataFrame(ALLP); PAIR.to_csv(f'{OUT}/val_paired.csv', index=False)
print('\nPaired comparisons (50 tests):'); display(PAIR)
with zipfile.ZipFile('/content/Validation_results.zip', 'w') as zf:
    for f in glob.glob(f'{OUT}/val_*.csv') + glob.glob(f'{OUT}/*.json'):
        zf.write(f, os.path.basename(f))
print('Saved to', OUT, '\nZip: /content/Validation_results.zip')
